# Create the Corpus Problem Lists

The notebook below creates the lists of documents for the corpus and datatype. This is used for iterating across the documents in the corpus when creating jobscripts that pull the known and unknown documents. To run this for a new set of data simply change the **corpus** and **data_type** parameters.

In [14]:
import sys
import pandas as pd

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from utils import get_base_location, build_metadata_df, apply_temp_doc_id
from read_and_write_docs import read_jsonl, read_rds

In [15]:
corpus      = "Wiki"
data_type   = "training"

# Set NAS so can run on Windows laptop seamlessly
nas_base_loc = get_base_location()

# --- set your args (strings) ---
nas_base_loc = "/Users/user/Documents/uni_work_offline"

known_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/{corpus}/known_raw.jsonl"
unknown_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/{corpus}/unknown_raw.jsonl"
metadata_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/metadata.rds"

save_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/{corpus}"

## Read Data

In [16]:
metadata = read_rds(metadata_loc)
filtered_metadata = metadata[metadata['corpus'] == corpus]

known = read_jsonl(known_loc)
unknown = read_jsonl(unknown_loc)

In [17]:
filtered_metadata

,problem,corpus,known_author,unknown_author
7272,142.196.88.228 vs 142.196.88.228,Wiki,142.196.88.228,142.196.88.228
7273,142.196.88.228 vs Aban1313,Wiki,142.196.88.228,Aban1313
7274,A_Man_In_Black vs A_Man_In_Black,Wiki,A_Man_In_Black,A_Man_In_Black
7275,A_Man_In_Black vs Bankhallbretherton,Wiki,A_Man_In_Black,Bankhallbretherton
7276,Aban1313 vs Aban1313,Wiki,Aban1313,Aban1313
...,...,...,...,...
7417,Haymaker vs HeadleyDown,Wiki,Haymaker,HeadleyDown
7418,HeadleyDown vs HeadleyDown,Wiki,HeadleyDown,HeadleyDown
7419,HeadleyDown vs Hipocrite,Wiki,HeadleyDown,Hipocrite
7420,Hipocrite vs Hipocrite,Wiki,Hipocrite,Hipocrite


## Create Agg Problem Level Metadata

In [18]:
agg_problems = filtered_metadata['problem'].drop_duplicates()

## Create Metadata

Quite a convoluted process.

In [19]:
# Build the dataframe
complete_metadata = build_metadata_df(filtered_metadata, known, unknown)

# Set blank text column for function to work
complete_metadata['text'] = ''

# Rename the known column and create the new doc_id
complete_metadata.rename(columns={"known_doc_id": "orig_doc_id"}, inplace=True)
complete_metadata = apply_temp_doc_id(complete_metadata)
complete_metadata.rename(columns={
    "orig_doc_id": "orig_known_doc_id",
    "doc_id": "known_doc_id",
    "unknown_doc_id": "orig_doc_id"
}, inplace=True)

# Do the same for the unknown
complete_metadata = apply_temp_doc_id(complete_metadata)
complete_metadata.rename(columns={
    "orig_doc_id": "orig_unknown_doc_id",
    "doc_id": "unknown_doc_id",
}, inplace=True)

# Sort columns
complete_metadata = complete_metadata[["sample_id", "problem", "corpus", "known_doc_id", "unknown_doc_id"]]

## View the data

In [20]:
complete_metadata.head()

,sample_id,problem,corpus,known_doc_id,unknown_doc_id
0,1,142.196.88.228 vs 142.196.88.228,Wiki,142_196_88_228_text_1,142_196_88_228_text_2
1,2,142.196.88.228 vs 142.196.88.228,Wiki,142_196_88_228_text_3,142_196_88_228_text_2
2,3,142.196.88.228 vs 142.196.88.228,Wiki,142_196_88_228_text_4,142_196_88_228_text_2
3,4,142.196.88.228 vs Aban1313,Wiki,142_196_88_228_text_1,aban1313_text_4
4,5,142.196.88.228 vs Aban1313,Wiki,142_196_88_228_text_3,aban1313_text_4


## Create the Document Lists

In [21]:
known_doc_list = pd.Series(complete_metadata["known_doc_id"].astype(str))
unknown_doc_list = pd.Series(complete_metadata["unknown_doc_id"].astype(str))
problem_doc_list = known_doc_list + ' vs ' + unknown_doc_list

## Get Number of Rows in the Dataset

This is used for the jobscript.

In [22]:
num_rows_for_jobscript = complete_metadata.shape[0]
print(f"Number of rows needed in jobscript: {num_rows_for_jobscript}")

Number of rows needed in jobscript: 450


## Save the Lists

In [23]:
print(f"Writing agg problem list - {len(agg_problems)} Aggregated Problems")
agg_problems.to_csv(f"{save_loc}/agg_problem_list.txt", index=False, header=False)
print(f"Writing problem doc list - {len(problem_doc_list)} Problems")
problem_doc_list.to_csv(f"{save_loc}/problem_doc_list.txt", index=False, header=False)
print(f"Writing known doc list - {len(known_doc_list)} Documents")
known_doc_list.to_csv(f"{save_loc}/known_doc_list.txt", index=False, header=False)
print(f"Writing unknown doc list - {len(unknown_doc_list)} Documents")
unknown_doc_list.to_csv(f"{save_loc}/unknown_doc_list.txt", index=False, header=False)
print("Wrote doc lists")

Writing agg problem list - 150 Aggregated Problems
Writing problem doc list - 450 Problems
Writing known doc list - 450 Documents
Writing unknown doc list - 450 Documents
Wrote doc lists
